In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
load_dotenv()


client = OpenAI()                      # reads OPENAI_API_KEY from the environment
MODEL = "gpt-4o-mini"

# response = client.chat.completions.create(
#     model=MODEL,
#     messages=[{"role": "user", "content": "Hi, my name is Ramesh and I study transformers."}],
# )
# print(response.choices[0].message.content)


Hello Ramesh! That's great to hear. Transformers have become a foundational model in natural language processing and other fields. What specific aspects of transformers are you studying? Are you focusing on their architecture, applications, or something else?


In [36]:
import uuid
import json
import math
import os
import time
from pathlib import Path

from pydantic import BaseModel

In [37]:
SESSIONS_DIR = Path("sessions")
MAX_EVENTS_IN_PROMPT = 5

class Session:
    def __init__(self, user_id: str, session_id: str | None = None):
        self.user_id = user_id                       # WHO the conversation is with
        self.id = session_id or uuid.uuid4().hex[:8]  # WHICH conversation this is
        self.events: list[dict] = []                 # WHAT has been said, in order
        self.state: dict = {}   


    def add_event(self, role: str, content: str) -> None:
        self.events.append({"role": role, "content": content, "time": time.time()})

    def save(self) -> None:
        SESSIONS_DIR.mkdir(exist_ok=True)
        (SESSIONS_DIR / f"{self.id}.json").write_text(json.dumps(self.__dict__, indent=2))

    
    @classmethod
    def load(cls, session_id: str) -> "Session":
        data = json.loads((SESSIONS_DIR / f"{session_id}.json").read_text())
        session = cls(data["user_id"], data["id"])
        session.events, session.state = data["events"], data["state"]
        return session
        
def history_as_messages(session: Session) -> list[dict]:
    """Turns stored events into the message list the model expects (most recent events only)."""
    recent = session.events[-MAX_EVENTS_IN_PROMPT:]
    return [{"role": e["role"], "content": e["content"]} for e in recent]

In [38]:
MEMORY_FILE = Path("memories.json")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")  # Ollama: nomic-embed-text

In [39]:
class Facts(BaseModel):
    facts: list[str]

def embed(text: str) -> list[float]:
    """Turns text into a vector. Texts with similar meaning get similar vectors."""
    return client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding

def similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity: about 1.0 for the same meaning, lower for unrelated texts."""
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

def load_store() -> dict:
    return json.loads(MEMORY_FILE.read_text()) if MEMORY_FILE.exists() else {}


def extract_memories(session: Session) -> list[str]:
    transcript = "\n".join(f"{e['role']}: {e['content']}" for e in session.events)
    completion = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "system", "content":
                   "List short, lasting facts about the user from this conversation that would help "
                   "plan future trips: preferences, constraints, home city. Skip one-off details."},
                  {"role": "user", "content": transcript}],
        response_format=Facts,
    )
    return completion.choices[0].message.parsed.facts
    
# 2. STORE: save each new fact with its embedding (skipping exact duplicates).
def save_memories(user_id: str, facts: list[str]) -> None:
    store = load_store()
    memories = store.setdefault(user_id, [])
    known = {m["text"] for m in memories}
    memories += [{"text": fact, "embedding": embed(fact)} for fact in facts if fact not in known]
    MEMORY_FILE.write_text(json.dumps(store))

# 3. RETRIEVE: find the stored facts closest in meaning to the new message.
def recall(user_id: str, query: str, k: int = 2) -> list[tuple[float, str]]:
    memories = load_store().get(user_id, [])
    if not memories:
        return []
    query_vector = embed(query)
    scored = [(similarity(query_vector, m["embedding"]), m["text"]) for m in memories]
    return sorted(scored, reverse=True)[:k]

In [40]:
# 4. USE: place the recalled facts in the instructions for this turn.
def chat(session: Session, text: str) -> str:
    recalled = recall(session.user_id, text)
    for score, fact in recalled:
        print(f"  recalled ({score:.2f}): {fact}")
    system = ("You are a friendly travel assistant.\n"
              "Facts you remember about this user from earlier conversations:\n"
              + "\n".join(f"- {fact}" for _, fact in recalled))
    session.add_event("user", text)
    messages = [{"role": "system", "content": system}] + history_as_messages(session)
    reply = client.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content
    session.add_event("assistant", reply)
    session.save()
    return reply

In [41]:
if __name__ == "__main__":
    # Conversation 1: the user mentions lasting facts in passing.
    first = Session(user_id="asha")
    print("assistant:", chat(first, "Hi! I'm Asha from Pune. I'm vegetarian and I always "
                                    "prefer a window seat. Thinking about Goa next month."))
    facts = extract_memories(first)  # runs after the conversation, not during it
    save_memories("asha", facts)
    print("extracted memories:", facts)

    # Conversation 2: a new session with an empty history, days later.
    second = Session(user_id="asha")
    print("\nassistant:", chat(second, "Can you suggest a good place for dinner in Jaipur?"))

  recalled (0.53): User's name is Asha.
  recalled (0.53): User is planning a trip to Goa next month.
assistant: Hi Asha! I'm excited to help you with your trip to Goa! Since you're a vegetarian, there are plenty of delicious options to explore there. Do you need recommendations for places to stay, things to do, or vegetarian restaurants? And don’t forget to secure that window seat!
extracted memories: ["User's name is Asha.", 'User is from Pune.', 'User is vegetarian.', 'User prefers a window seat when traveling.', 'User is planning a trip to Goa next month.']
  recalled (0.23): User is planning a trip to Goa next month.
  recalled (0.23): User is from Pune.

assistant: Certainly! Jaipur offers a variety of dining options. One popular choice is **Chokhi Dhani**, which provides a traditional Rajasthani experience with authentic cuisine, cultural performances, and a vibrant atmosphere. If you're looking for something more upscale, **Suvarna Mahal** at the Rambagh Palace is known for its

In [25]:
apple_vector = embed("i like apples")

mango_vector = embed("i like mangoes")

space_vector = embed("i like to travel to mars")

In [24]:
mango_vector

array([[ 4.51049805e-02],
       [-3.72314453e-02],
       [-3.17687988e-02],
       ...,
       [ 2.92816162e-02],
       [-9.83476639e-05],
       [ 4.28466797e-02]], shape=(1536, 1))

In [23]:
space_vector

array([[-0.01084137],
       [-0.03469849],
       [ 0.00324631],
       ...,
       [ 0.03149414],
       [-0.00375557],
       [ 0.01160431]], shape=(1536, 1))

In [28]:
apple_vector = np.array(apple_vector).reshape(1, -1)

mango_vector = np.array(mango_vector).reshape(1, -1)

space_vector =  np.array(space_vector).reshape(1, -1)

In [29]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(apple_vector, mango_vector)
print(f"Cosine Similarity: {similarity[0][0]:.4f}")

Cosine Similarity: 0.5991


In [30]:
similarity = cosine_similarity(apple_vector, space_vector)
print(f"Cosine Similarity: {similarity[0][0]:.4f}")

Cosine Similarity: 0.3734


## Memory Store like Mem0

In [43]:
# !pip install mem0ai

In [45]:
load_dotenv()

True

In [46]:
import os
from mem0 import MemoryClient

In [48]:
# os.getenv("MEM0_API_KEY")

In [49]:
USER_ID = "ram"

In [51]:
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))

In [57]:
# Add a memory
messages = [
    {"role": "user", "content": "I'm a vegetarian and allergic to nuts."},
    {"role": "assistant", "content": "Got it! I'll remember your dietary preferences."},
    {"role": "user", "content": "I am working as a software engineer in hyderabad and i like to travel to mountains"},
]
client.add(messages, user_id=USER_ID)

{'event_id': '7ab8e91c-49dd-43ba-b0bc-78331b2a2419', 'status': 'PENDING'}

In [53]:
# Search memories
results = client.search(
    "What are my dietary restrictions?",
    filters={"user_id": USER_ID}
)

In [54]:
results

{'results': [{'id': '32c5c7b7-2d28-4a20-bd3f-9f79b5b319f7',
   'memory': 'User is a vegetarian and is allergic to nuts',
   'user_id': 'ram',
   'agent_id': None,
   'app_id': None,
   'run_id': None,
   'score': 0.2201,
   'score_breakdown': {'semantic': 0.5502, 'bm25': 0.0, 'entity': 0.0},
   'metadata': {},
   'categories': ['food', 'health', 'user_preferences'],
   'created_at': '2026-10-24T15:57:47+00:00',
   'updated_at': '2026-10-24T15:57:53.279643+00:00',
   'expiration_date': None}]}

In [55]:
results = client.search(
    "What is my profession??? and what is my name?",
    filters={"user_id": USER_ID}
)

In [56]:
results

{'results': [{'id': '32c5c7b7-2d28-4a20-bd3f-9f79b5b319f7',
   'memory': 'User is a vegetarian and is allergic to nuts',
   'user_id': 'ram',
   'agent_id': None,
   'app_id': None,
   'run_id': None,
   'score': 0.0909,
   'score_breakdown': {'semantic': 0.2273, 'bm25': 0.0, 'entity': 0.0},
   'metadata': {},
   'categories': ['food', 'health', 'user_preferences'],
   'created_at': '2026-10-24T15:57:47+00:00',
   'updated_at': '2026-10-24T15:57:53.279643+00:00',
   'expiration_date': None}]}

In [58]:
results = client.search(
    "What is my profession??? and what is my name?",
    filters={"user_id": USER_ID}
)

In [59]:
results

{'results': [{'id': '46e13946-1e5a-4cd1-99b5-465525f8bfa0',
   'memory': 'User works as a software engineer in Hyderabad',
   'user_id': 'ram',
   'agent_id': None,
   'app_id': None,
   'run_id': None,
   'score': 0.1736,
   'score_breakdown': {'semantic': 0.434, 'bm25': 0.0, 'entity': 0.0},
   'metadata': {},
   'categories': ['professional_details'],
   'created_at': '2026-10-24T16:03:25+00:00',
   'updated_at': '2026-10-24T16:03:33.483118+00:00',
   'expiration_date': None},
  {'id': '6433460e-d4ed-4189-85e0-365d02afd470',
   'memory': 'User likes traveling to mountains',
   'user_id': 'ram',
   'agent_id': None,
   'app_id': None,
   'run_id': None,
   'score': 0.0991,
   'score_breakdown': {'semantic': 0.2477, 'bm25': 0.0, 'entity': 0.0},
   'metadata': {},
   'categories': ['hobbies', 'travel', 'user_preferences'],
   'created_at': '2026-10-24T16:03:25+00:00',
   'updated_at': '2026-10-24T16:03:33.483118+00:00',
   'expiration_date': None},
  {'id': '32c5c7b7-2d28-4a20-bd3f-9f79